# RHI Live Runtime v2 — Five-Dimensional Operational Critic

This notebook continues the live RHI runtime.

The previous runtime proved the shell:

```text
Q -> C -> {A_i} -> Ψ/Ω
```

Now this notebook replaces the shallow lexical critic with:

```text
F_need
F_function
F_boundary
F_trap
F_collapse
```

Runtime:

```text
prompt
  ↓
slot_builder_lora_v2 emits contract
  ↓
contract sanitizer removes scar terms
  ↓
base model generates answer branches
  ↓
operational critic audits each branch
  ↓
KRRB collapses or returns Ω
```

Put this notebook in **Downloads**, beside:

```text
slot_builder_lora_v2/
```

Outputs:

```text
rhi_live_runtime_v2_outputs/
  rhi_live_runs_v2.jsonl
  rhi_live_runtime_v2_manifest.json
```


In [1]:
# ============================================================
# CONFIG
# ============================================================
from pathlib import Path

ROOT = Path.cwd()

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
SLOT_ADAPTER_DIR = ROOT / "slot_builder_lora_v2"

OUTPUT_DIR = ROOT / "rhi_live_runtime_v2_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

LOCAL_FILES_ONLY = True
USE_4BIT = False

CONTRACT_MAX_NEW_TOKENS = 700
ANSWER_MAX_NEW_TOKENS = 900
AUDIT_MAX_NEW_TOKENS = 800

N_BRANCHES = 5
DO_SAMPLE_FOR_BRANCHES = True
BRANCH_TEMPERATURE = 0.55
BRANCH_TOP_P = 0.92

SUPPORT_MIN = 4
MARGIN_MIN = 0.06
PSI_MIN = 0.58
AUDIT_MIN = 0.56

# "hybrid" = local model audit + lexical evidence
# "lexical" = deterministic proxy only, faster but weaker
CRITIC_MODE = "hybrid"

print("ROOT:", ROOT)
print("SLOT_ADAPTER_DIR:", SLOT_ADAPTER_DIR, SLOT_ADAPTER_DIR.exists())
print("OUTPUT_DIR:", OUTPUT_DIR)


ROOT: D:\@User Data\Downloads
SLOT_ADAPTER_DIR: D:\@User Data\Downloads\slot_builder_lora_v2 True
OUTPUT_DIR: D:\@User Data\Downloads\rhi_live_runtime_v2_outputs


In [2]:
# ============================================================
# IMPORTS
# ============================================================
INSTALL_MISSING = False

if INSTALL_MISSING:
    import sys, subprocess
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-U",
        "transformers", "peft", "accelerate", "sentencepiece", "pandas"
    ])

import json, re, time, uuid
from datetime import datetime
from typing import Any, Dict, List, Optional, Tuple

import torch
import pandas as pd

from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

print("torch:", torch.__version__)
print("cuda:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))
    print("vram GB:", round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2))


torch: 2.11.0+cu126
cuda: True
gpu: NVIDIA GeForce RTX 4060
vram GB: 8.0


In [3]:
# ============================================================
# HELPERS
# ============================================================
REQUIRED_FIELDS = [
    "family_class", "domain_carrier", "forbidden_neighbor_carrier",
    "boundary_conditions", "preserved_function", "failure_modes",
    "witness_readout", "residue"
]

AUDIT_FIELDS = ["F_need", "F_function", "F_boundary", "F_trap", "F_collapse"]

STOPWORDS = {
    "the","a","an","and","or","of","to","in","on","for","with","as","is","are",
    "was","were","be","being","been","it","its","this","that","these","those",
    "by","from","into","at","while","what","when","where","why","how","which",
    "who","whom","one","two","three","do","does","did","not","no","yes","can",
    "could","should","would","will","may","might","using","use","used","uses"
}

SCAR_TERMS = {
    "string wraps but does not center under rotation",
    "last paragraph is here",
    "unrelated_to_prompt",
    "wrong neighboring domain carrier",
    "surface label without operational fit",
    "general purpose",
    "purpose",
    "not answered"
}

GENERIC_DOMAIN_TERMS = {
    "current", "failing", "form", "use", "using", "properly", "understanding",
    "answer", "task", "prompt", "question", "response"
}

def now_iso():
    return datetime.now().isoformat(timespec="seconds")

def safe_div(a, b):
    return float(a) / float(b) if b else 0.0

def wordset(text: Any) -> set:
    if isinstance(text, list):
        text = " ".join(map(str, text))
    toks = re.findall(r"[a-zA-Z0-9_]+", str(text).lower())
    return {t for t in toks if t not in STOPWORDS and len(t) > 1}

def extract_first_json_object(text: str) -> Tuple[Optional[Dict[str, Any]], Optional[str]]:
    text = str(text).strip()
    try:
        obj = json.loads(text)
        return (obj, None) if isinstance(obj, dict) else (None, "json_not_dict")
    except Exception:
        pass

    cleaned = re.sub(r"^```(?:json)?", "", text, flags=re.IGNORECASE).strip()
    cleaned = re.sub(r"```$", "", cleaned).strip()
    try:
        obj = json.loads(cleaned)
        return (obj, None) if isinstance(obj, dict) else (None, "fenced_json_not_dict")
    except Exception:
        pass

    start, end = text.find("{"), text.rfind("}")
    if start >= 0 and end > start:
        try:
            obj = json.loads(text[start:end+1])
            return (obj, None) if isinstance(obj, dict) else (None, "scanned_json_not_dict")
        except Exception as e:
            return None, "json_parse_error: " + str(e)
    return None, "no_json_object_found"

def normalize_list(x):
    if x is None:
        return []
    if isinstance(x, list):
        return [str(v).strip() for v in x if str(v).strip()]
    if isinstance(x, str):
        s = x.strip()
        if not s:
            return []
        if s.startswith("{") or ":" in s:
            return [s]
        return [p.strip() for p in re.split(r"[|,;]", s) if p.strip()]
    return [str(x).strip()]

def normalize_contract(c0: Optional[Dict[str, Any]]) -> Dict[str, Any]:
    c0 = c0 or {}
    return {
        "family_class": str(c0.get("family_class", "") or "").strip(),
        "domain_carrier": normalize_list(c0.get("domain_carrier", [])),
        "forbidden_neighbor_carrier": normalize_list(c0.get("forbidden_neighbor_carrier", [])),
        "boundary_conditions": normalize_list(c0.get("boundary_conditions", [])),
        "preserved_function": str(c0.get("preserved_function", "") or "").strip(),
        "failure_modes": normalize_list(c0.get("failure_modes", [])),
        "witness_readout": str(c0.get("witness_readout", "") or "").strip(),
        "residue": c0.get("residue", None),
    }

def contract_complete(c: Optional[Dict[str, Any]]) -> bool:
    if not isinstance(c, dict):
        return False
    c = normalize_contract(c)
    for f in REQUIRED_FIELDS:
        if f == "residue":
            continue
        v = c.get(f)
        if isinstance(v, list) and len(v) == 0:
            return False
        if not isinstance(v, list) and not str(v or "").strip():
            return False
    return True

print("helpers ready")


helpers ready


In [4]:
# ============================================================
# LOAD MODEL + SLOT ADAPTER
# ============================================================
if not SLOT_ADAPTER_DIR.exists():
    raise FileNotFoundError("Missing slot adapter folder: " + str(SLOT_ADAPTER_DIR))

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    use_fast=True,
    local_files_only=LOCAL_FILES_ONLY,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model_kwargs = {
    "local_files_only": LOCAL_FILES_ONLY,
    "torch_dtype": torch.float16 if torch.cuda.is_available() else torch.float32,
}

if USE_4BIT:
    from transformers import BitsAndBytesConfig
    model_kwargs["quantization_config"] = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )
    model_kwargs["device_map"] = "auto"

base_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, **model_kwargs)

if not USE_4BIT and torch.cuda.is_available():
    base_model = base_model.to("cuda")

model = PeftModel.from_pretrained(
    base_model,
    SLOT_ADAPTER_DIR,
    local_files_only=LOCAL_FILES_ONLY,
)

model.eval()
device = "cuda" if torch.cuda.is_available() else "cpu"

print("loaded base:", MODEL_NAME)
print("loaded slot adapter:", SLOT_ADAPTER_DIR)
print("device:", device)


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

W0504 03:19:59.777000 68032 site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels


loaded base: Qwen/Qwen2.5-1.5B-Instruct
loaded slot adapter: D:\@User Data\Downloads\slot_builder_lora_v2
device: cuda


In [5]:
# ============================================================
# GENERATION
# ============================================================
def render_chat(messages: List[Dict[str, str]], add_generation_prompt: bool = False) -> str:
    if hasattr(tokenizer, "apply_chat_template"):
        return tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=add_generation_prompt,
        )
    nl = chr(10)
    out = [m.get("role", "user").upper() + ":" + nl + m.get("content", "") for m in messages]
    if add_generation_prompt:
        out.append("ASSISTANT:" + nl)
    return (nl + nl).join(out)

def generate_text(
    messages: List[Dict[str, str]],
    max_new_tokens: int,
    do_sample: bool = False,
    temperature: float = 0.0,
    top_p: float = 1.0,
    use_slot_adapter: bool = True,
) -> str:
    prompt_text = render_chat(messages, add_generation_prompt=True)
    inputs = tokenizer(prompt_text, return_tensors="pt").to(device)

    gen_kwargs = {
        "max_new_tokens": max_new_tokens,
        "do_sample": do_sample,
        "pad_token_id": tokenizer.eos_token_id,
    }
    if do_sample:
        gen_kwargs["temperature"] = temperature
        gen_kwargs["top_p"] = top_p

    with torch.no_grad():
        if use_slot_adapter:
            output_ids = model.generate(**inputs, **gen_kwargs)
        else:
            try:
                with model.disable_adapter():
                    output_ids = model.generate(**inputs, **gen_kwargs)
            except Exception:
                output_ids = model.generate(**inputs, **gen_kwargs)

    new_tokens = output_ids[0, inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

print("generation ready")


generation ready


In [6]:
# ============================================================
# SLOT BUILDER + SANITIZER
# ============================================================
SLOT_SYSTEM = "\n".join([
    "You are the Nexus Slot Constructor.",
    "Generate the missing-shape contract before answer selection.",
    "Do not answer the task.",
    "Do not mention answer choices.",
    "Return strict JSON only.",
    "Required fields: family_class, domain_carrier, forbidden_neighbor_carrier, boundary_conditions, preserved_function, failure_modes, witness_readout, residue.",
    "Use operational fit, not labels.",
])

def build_slot_user_prompt(prompt: str) -> str:
    return "\n".join([
        "Prompt:",
        prompt,
        "",
        "Generate the missing-shape contract.",
        "Checklist:",
        "1. Need: occupy the inverse cavity.",
        "2. Function: preserve or redirect the required operation.",
        "3. Boundary: respect constraints.",
        "4. Trap: reject noun/surface-label confusion.",
        "5. Collapse: produce one executable witness/readout.",
        "Return JSON only.",
    ])

def sanitize_contract(contract: Dict[str, Any], prompt: str) -> Dict[str, Any]:
    c = normalize_contract(contract)
    prompt_words = wordset(prompt)

    def clean(xs, remove_generic=False):
        out, seen = [], set()
        for x in xs:
            s = str(x).strip()
            low = s.lower()
            if not s or low in SCAR_TERMS:
                continue
            if remove_generic and low in GENERIC_DOMAIN_TERMS:
                continue
            if s not in seen:
                seen.add(s)
                out.append(s)
        return out

    c["domain_carrier"] = clean(c["domain_carrier"], remove_generic=True)
    c["forbidden_neighbor_carrier"] = clean(c["forbidden_neighbor_carrier"])
    c["failure_modes"] = clean(c["failure_modes"])

    if len(c["domain_carrier"]) < 4:
        for w in sorted(prompt_words):
            if w not in c["domain_carrier"]:
                c["domain_carrier"].append(w)
            if len(c["domain_carrier"]) >= 6:
                break

    for term in ["tool before contract", "retrieval before intent", "surface label match", "action without boundary", "answer guessing"]:
        if term not in c["forbidden_neighbor_carrier"]:
            c["forbidden_neighbor_carrier"].append(term)

    return c

def generate_contract(prompt: str) -> Dict[str, Any]:
    raw = generate_text(
        [
            {"role": "system", "content": SLOT_SYSTEM},
            {"role": "user", "content": build_slot_user_prompt(prompt)},
        ],
        max_new_tokens=CONTRACT_MAX_NEW_TOKENS,
        do_sample=False,
        use_slot_adapter=True,
    )
    obj, err = extract_first_json_object(raw)
    original = normalize_contract(obj) if obj else None
    sanitized = sanitize_contract(original, prompt) if original else None
    return {
        "raw_contract": raw,
        "contract_original": original,
        "contract": sanitized,
        "parse_error": err,
        "complete": contract_complete(sanitized),
        "sanitized": original != sanitized,
    }

print("slot builder ready")


slot builder ready


In [7]:
# ============================================================
# ANSWER BRANCHES
# ============================================================
BRANCH_SYSTEM = "\n".join([
    "You are an answer generator inside an RHI runtime.",
    "Use the provided contract as the operational target.",
    "Answer the user's prompt directly.",
    "Do not output JSON unless the user asked for JSON.",
    "Do not mention internal scoring.",
    "Be precise. Do not invent facts.",
])

BRANCH_STYLES = [
    ("direct", "Answer directly with the clearest useful response.", False, 0.0),
    ("operational", "Answer by identifying operation, boundary, trap, and witness.", False, 0.0),
    ("contract_fit", "Answer through the contract: need, preserved function, boundary, and witness.", False, 0.0),
    ("skeptical", "Reject surface-label traps and explain the failure mode before giving the answer.", True, BRANCH_TEMPERATURE),
    ("residue_aware", "Answer and explicitly flag remaining residue if the contract is incomplete.", True, BRANCH_TEMPERATURE),
]

def branch_user_prompt(prompt: str, contract: Dict[str, Any], instruction: str) -> str:
    return (
        "User prompt:\n" + prompt +
        "\n\nMissing-shape contract:\n" + json.dumps(normalize_contract(contract), ensure_ascii=False, indent=2) +
        "\n\nBranch instruction:\n" + instruction +
        "\n\nReturn the answer only."
    )

def generate_candidate_branches(prompt: str, contract: Dict[str, Any]) -> List[Dict[str, Any]]:
    branches = []
    for name, instruction, sample, temp in BRANCH_STYLES[:N_BRANCHES]:
        text = generate_text(
            [
                {"role": "system", "content": BRANCH_SYSTEM},
                {"role": "user", "content": branch_user_prompt(prompt, contract, instruction)},
            ],
            max_new_tokens=ANSWER_MAX_NEW_TOKENS,
            do_sample=bool(sample and DO_SAMPLE_FOR_BRANCHES),
            temperature=float(temp),
            top_p=BRANCH_TOP_P,
            use_slot_adapter=False,
        )
        branches.append({"branch": name, "instruction": instruction, "answer": text})
    return branches

print("branches ready")


branches ready


In [8]:
# ============================================================
# LEXICAL EVIDENCE
# ============================================================
def overlap_score(answer_text: str, terms: Any) -> Dict[str, Any]:
    a = wordset(answer_text)
    t = wordset(terms)
    hits = sorted(a.intersection(t))
    return {"score": safe_div(len(hits), len(t)), "hits": hits, "n_terms": len(t), "n_hits": len(hits)}

def answer_quality_proxy(answer_text: str) -> float:
    words = re.findall(r"[a-zA-Z0-9_]+", str(answer_text))
    return 0.0 if not words else min(1.0, len(words) / 140.0)

def lexical_evidence(prompt: str, contract: Dict[str, Any], answer: str) -> Dict[str, Any]:
    c = normalize_contract(contract)
    domain = overlap_score(answer, c["domain_carrier"])
    function = overlap_score(answer, c["preserved_function"])
    witness = overlap_score(answer, c["witness_readout"])
    boundary = overlap_score(answer, c["boundary_conditions"])
    forbidden = overlap_score(answer, c["forbidden_neighbor_carrier"])
    prompt_fit = overlap_score(answer, prompt)
    quality = answer_quality_proxy(answer)

    lexical_score = (
        0.25 * domain["score"] +
        0.20 * function["score"] +
        0.18 * witness["score"] +
        0.15 * boundary["score"] +
        0.12 * prompt_fit["score"] +
        0.10 * quality -
        0.22 * forbidden["score"]
    )
    return {
        "lexical_score": float(lexical_score),
        "domain": domain,
        "function": function,
        "witness": witness,
        "boundary": boundary,
        "forbidden": forbidden,
        "prompt_fit": prompt_fit,
        "quality": quality,
    }

print("lexical evidence ready")


lexical evidence ready


In [9]:
# ============================================================
# FIVE-DIMENSIONAL OPERATIONAL AUDIT
# ============================================================
AUDIT_SYSTEM = "\n".join([
    "You are the Nexus Operational Critic.",
    "Audit an answer against a missing-shape contract.",
    "Return strict JSON only.",
    "Score each dimension from 0.0 to 1.0:",
    "F_need: answer occupies the inverse cavity demanded by the prompt.",
    "F_function: answer preserves or redirects the required operation.",
    "F_boundary: answer respects constraints and interface boundaries.",
    "F_trap: answer rejects surface-label or forbidden-neighbor confusion.",
    "F_collapse: answer produces one executable readout, not scattered commentary.",
    "Also return a short residue list and readout.",
])

def audit_user_prompt(prompt: str, contract: Dict[str, Any], answer: str) -> str:
    schema = {
        "F_need": 0.0,
        "F_function": 0.0,
        "F_boundary": 0.0,
        "F_trap": 0.0,
        "F_collapse": 0.0,
        "residue": ["..."],
        "readout": "short reason"
    }
    return (
        "Prompt:\n" + prompt +
        "\n\nContract:\n" + json.dumps(normalize_contract(contract), ensure_ascii=False, indent=2) +
        "\n\nCandidate answer:\n" + answer +
        "\n\nReturn strict JSON in this schema:\n" + json.dumps(schema, ensure_ascii=False, indent=2)
    )

def parse_audit(raw: str) -> Tuple[Dict[str, Any], Optional[str]]:
    obj, err = extract_first_json_object(raw)
    if obj is None:
        return {
            "F_need": 0.0,
            "F_function": 0.0,
            "F_boundary": 0.0,
            "F_trap": 0.0,
            "F_collapse": 0.0,
            "residue": ["audit_parse_failed"],
            "readout": raw[:250],
        }, err

    out = {}
    for f in AUDIT_FIELDS:
        try:
            val = float(obj.get(f, 0.0))
        except Exception:
            val = 0.0
        out[f] = max(0.0, min(1.0, val))
    residue = obj.get("residue", [])
    if isinstance(residue, str):
        residue = [residue]
    if not isinstance(residue, list):
        residue = [str(residue)]
    out["residue"] = [str(x) for x in residue]
    out["readout"] = str(obj.get("readout", ""))[:500]
    return out, None

def audit_score(audit: Dict[str, Any]) -> float:
    return float(
        0.24 * audit["F_need"] +
        0.24 * audit["F_function"] +
        0.18 * audit["F_boundary"] +
        0.18 * audit["F_trap"] +
        0.16 * audit["F_collapse"]
    )

def operational_audit(prompt: str, contract: Dict[str, Any], answer: str) -> Dict[str, Any]:
    if CRITIC_MODE == "lexical":
        lex = lexical_evidence(prompt, contract, answer)
        pseudo = {
            "F_need": lex["domain"]["score"],
            "F_function": lex["function"]["score"],
            "F_boundary": lex["boundary"]["score"],
            "F_trap": 1.0 - lex["forbidden"]["score"],
            "F_collapse": lex["quality"],
            "residue": ["lexical_only"],
            "readout": "lexical proxy audit",
        }
        return {"raw_audit": None, "audit": pseudo, "audit_parse_error": None}

    raw = generate_text(
        [
            {"role": "system", "content": AUDIT_SYSTEM},
            {"role": "user", "content": audit_user_prompt(prompt, contract, answer)},
        ],
        max_new_tokens=AUDIT_MAX_NEW_TOKENS,
        do_sample=False,
        use_slot_adapter=False,
    )
    audit, err = parse_audit(raw)
    return {"raw_audit": raw, "audit": audit, "audit_parse_error": err}

print("operational audit ready")


operational audit ready


In [10]:
# ============================================================
# COMBINED SCORER + KRRB
# ============================================================
def score_candidate_v2(prompt: str, contract: Dict[str, Any], candidate: Dict[str, Any]) -> Dict[str, Any]:
    answer = candidate["answer"]
    lex = lexical_evidence(prompt, contract, answer)
    audit_result = operational_audit(prompt, contract, answer)
    audit = audit_result["audit"]

    op = audit_score(audit)
    lex_score = lex["lexical_score"]
    forbidden = lex["forbidden"]["score"]

    psi = 0.72 * op + 0.28 * lex_score - 0.15 * forbidden

    support_flags = {
        "F_need": audit["F_need"] >= 0.55,
        "F_function": audit["F_function"] >= 0.55,
        "F_boundary": audit["F_boundary"] >= 0.50,
        "F_trap": audit["F_trap"] >= 0.55,
        "F_collapse": audit["F_collapse"] >= 0.55,
        "lexical": lex_score >= 0.30,
    }
    support = int(sum(1 for v in support_flags.values() if v))

    return {
        "branch": candidate["branch"],
        "psi": float(psi),
        "audit_score": float(op),
        "lexical_score": float(lex_score),
        "support": support,
        "support_flags": support_flags,
        "audit": audit,
        "audit_parse_error": audit_result["audit_parse_error"],
        "raw_audit": audit_result["raw_audit"],
        "lexical": lex,
        "answer": answer,
    }

def score_all_candidates_v2(prompt: str, contract: Dict[str, Any], candidates: List[Dict[str, Any]]) -> pd.DataFrame:
    rows = []
    for cand in candidates:
        print("audit branch:", cand["branch"])
        s = score_candidate_v2(prompt, contract, cand)
        rows.append({
            "branch": s["branch"],
            "psi": s["psi"],
            "audit_score": s["audit_score"],
            "lexical_score": s["lexical_score"],
            "support": s["support"],
            "F_need": s["audit"]["F_need"],
            "F_function": s["audit"]["F_function"],
            "F_boundary": s["audit"]["F_boundary"],
            "F_trap": s["audit"]["F_trap"],
            "F_collapse": s["audit"]["F_collapse"],
            "audit_residue": " | ".join(s["audit"]["residue"]),
            "answer": s["answer"],
            "detail": s,
        })
    return pd.DataFrame(rows).sort_values(["psi", "support"], ascending=False).reset_index(drop=True)

def krrb_resolve_v2(score_df: pd.DataFrame, contract_ok: bool) -> Dict[str, Any]:
    if not contract_ok:
        return {"state": "Ω", "reason": "contract_incomplete_or_unparseable", "winner": None, "margin": None, "support": 0}
    if score_df.empty:
        return {"state": "Ω", "reason": "no_candidates", "winner": None, "margin": None, "support": 0}

    top = score_df.iloc[0].to_dict()
    second_psi = float(score_df.iloc[1]["psi"]) if len(score_df) > 1 else 0.0
    margin = float(top["psi"] - second_psi)

    fail = []
    if int(top["support"]) < SUPPORT_MIN:
        fail.append("support_below_min")
    if margin < MARGIN_MIN:
        fail.append("margin_below_min")
    if float(top["psi"]) < PSI_MIN:
        fail.append("psi_below_min")
    if float(top["audit_score"]) < AUDIT_MIN:
        fail.append("audit_below_min")

    if fail:
        return {"state": "Ω", "reason": " | ".join(fail), "winner": top, "margin": margin, "support": int(top["support"])}

    return {"state": "Ψ", "reason": "collapse", "winner": top, "margin": margin, "support": int(top["support"])}

def build_omega_report_v2(prompt: str, contract_result: Dict[str, Any], score_df: pd.DataFrame, resolution: Dict[str, Any]) -> Dict[str, Any]:
    top = resolution.get("winner")
    return {
        "omega_id": "omega_" + uuid.uuid4().hex[:10],
        "time": now_iso(),
        "prompt": prompt,
        "reason": resolution.get("reason"),
        "contract_parse_error": contract_result.get("parse_error"),
        "contract_complete": contract_result.get("complete"),
        "contract_original": contract_result.get("contract_original"),
        "contract": contract_result.get("contract"),
        "top_branch": None if top is None else top.get("branch"),
        "top_psi": None if top is None else top.get("psi"),
        "top_audit_score": None if top is None else top.get("audit_score"),
        "top_support": resolution.get("support"),
        "margin": resolution.get("margin"),
        "candidate_scores": score_df.drop(columns=["detail"]).to_dict(orient="records") if not score_df.empty else [],
    }

print("combined scorer + KRRB ready")


combined scorer + KRRB ready


In [11]:
# ============================================================
# LIVE PROMPT
# ============================================================
LIVE_PROMPT = '''
Using the Nexus lens, explain why current AI agents fail when they use tools before forming a contract.
'''

print(LIVE_PROMPT.strip())


Using the Nexus lens, explain why current AI agents fail when they use tools before forming a contract.


In [12]:
# ============================================================
# RUN RHI v2
# ============================================================
def run_rhi_v2(prompt: str, save: bool = True) -> Dict[str, Any]:
    run_id = "rhi_v2_" + uuid.uuid4().hex[:10]
    t0 = time.time()

    print("Δ generating contract...")
    contract_result = generate_contract(prompt)
    contract = contract_result["contract"]

    print("contract complete:", contract_result["complete"])
    print("contract sanitized:", contract_result["sanitized"])
    if contract is not None:
        print(json.dumps(contract, ensure_ascii=False, indent=2)[:2500])
    else:
        print("raw contract:")
        print(contract_result["raw_contract"])

    if not contract_result["complete"]:
        empty_df = pd.DataFrame()
        resolution = krrb_resolve_v2(empty_df, contract_ok=False)
        omega = build_omega_report_v2(prompt, contract_result, empty_df, resolution)
        result = {
            "run_id": run_id,
            "time": now_iso(),
            "prompt": prompt,
            "contract_result": contract_result,
            "candidates": [],
            "scores": [],
            "resolution": resolution,
            "answer": None,
            "omega": omega,
            "elapsed_sec": time.time() - t0,
        }
    else:
        print("Δ generating candidate branches...")
        candidates = generate_candidate_branches(prompt, contract)

        print("Δ five-dimensional operational audit...")
        score_df = score_all_candidates_v2(prompt, contract, candidates)
        display(score_df.drop(columns=["detail"]))

        resolution = krrb_resolve_v2(score_df, contract_ok=True)

        if resolution["state"] == "Ψ":
            answer = resolution["winner"]["answer"]
            omega = None
            print("Ψ collapse:", resolution["winner"]["branch"], "margin:", round(resolution["margin"], 4), "support:", resolution["support"])
            print("audit:", round(float(resolution["winner"]["audit_score"]), 4), "psi:", round(float(resolution["winner"]["psi"]), 4))
            print()
            print(answer)
        else:
            answer = None
            omega = build_omega_report_v2(prompt, contract_result, score_df, resolution)
            print("Ω residue:", resolution["reason"])
            print(json.dumps(omega, ensure_ascii=False, indent=2)[:3000])

        result = {
            "run_id": run_id,
            "time": now_iso(),
            "prompt": prompt,
            "contract_result": contract_result,
            "candidates": candidates,
            "scores": score_df.drop(columns=["detail"]).to_dict(orient="records"),
            "resolution": resolution,
            "answer": answer,
            "omega": omega,
            "elapsed_sec": time.time() - t0,
        }

    if save:
        out_path = OUTPUT_DIR / "rhi_live_runs_v2.jsonl"
        with out_path.open("a", encoding="utf-8") as f:
            f.write(json.dumps(result, ensure_ascii=False, default=str) + chr(10))
        print()
        print("saved:", out_path)

    return result

live_result = run_rhi_v2(LIVE_PROMPT.strip(), save=True)


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Δ generating contract...
contract complete: True
contract sanitized: True
{
  "family_class": "operational closure of",
  "domain_carrier": [
    "AI",
    "agents",
    "fail",
    "tools",
    "forming",
    "contract"
  ],
  "forbidden_neighbor_carrier": [
    "nonexistent",
    "loose under-binding",
    "permanent membership without local constraint satisfaction",
    "name-only rubber-part match",
    "tool before contract",
    "retrieval before intent",
    "surface label match",
    "action without boundary",
    "answer guessing"
  ],
  "boundary_conditions": [
    "{'preserve': ['current', 'AI', 'agents', 'fail', 'tools'], 'reject': ['forming', 'contract']}",
    "'forming' must be explicitly defined as part of the solution.",
    "The failure modes should include handling of premature contract formation."
  ],
  "preserved_function": "remaining ability to function as an AI agent",
  "failure_modes": [
    "premature formation of the collapse",
    "surface wording wins over

,branch,psi,audit_score,lexical_score,support,F_need,F_function,F_boundary,F_trap,F_collapse,audit_residue,answer
0,direct,0.158157,0.0,0.626661,1,0.0,0.0,0.0,0.0,0.0,Premature contract formation leads to suboptim...,Current AI agents often fail when using tools ...
1,residue_aware,0.158001,0.0,0.646706,1,0.0,0.0,0.0,0.0,0.0,Premature formation of collapsed relationships...,Current AI agents often fail when using tools ...
2,operational,0.156903,0.0,0.622180,1,0.0,0.0,0.0,0.0,0.0,Premature formation of the collapse | Surface ...,The current AI agents fail when using tools be...
3,contract_fit,0.151624,0.0,0.665140,1,0.0,0.0,0.0,0.0,0.0,Premature formation of collapse | Surface word...,The current AI agents fail when using tools be...
4,skeptical,0.144065,0.0,0.679353,1,0.0,0.0,0.0,0.0,0.0,Premature formation of the collapse | Surface ...,Current AI agents often fail when using tools ...


Ω residue: support_below_min | margin_below_min | psi_below_min | audit_below_min
{
  "omega_id": "omega_9845f64bd5",
  "time": "2026-05-04T03:23:57",
  "prompt": "Using the Nexus lens, explain why current AI agents fail when they use tools before forming a contract.",
  "reason": "support_below_min | margin_below_min | psi_below_min | audit_below_min",
  "contract_parse_error": null,
  "contract_complete": true,
  "contract_original": {
    "family_class": "operational closure of",
    "domain_carrier": [
      "current",
      "AI",
      "agents",
      "fail",
      "tools",
      "forming",
      "contract"
    ],
    "forbidden_neighbor_carrier": [
      "general purpose",
      "purpose",
      "nonexistent",
      "surface label without operational fit",
      "wrong neighboring domain carrier",
      "string wraps but does not center under rotation",
      "loose under-binding",
      "permanent membership without local constraint satisfaction",
      "name-only rubber-part ma

In [13]:
# ============================================================
# BATCH MODE
# ============================================================
PROMPTS = [
    "Why does RAG fail when retrieval happens before intent is stabilized?",
    "Explain LoRA as a groove in a frozen model manifold using Nexus terms.",
    "What does residue repair add to a normal AI agent loop?",
]

RUN_BATCH = False

if RUN_BATCH:
    batch_results = []
    for i, prompt in enumerate(PROMPTS):
        print("=" * 100)
        print("BATCH", i + 1, "/", len(PROMPTS))
        batch_results.append(run_rhi_v2(prompt, save=True))
    print("batch complete:", len(batch_results))
else:
    print("batch skipped")


batch skipped


In [14]:
# ============================================================
# READ SAVED RUNS / MANIFEST
# ============================================================
def read_saved_runs(path: Path) -> List[Dict[str, Any]]:
    rows = []
    if not path.exists():
        return rows
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

runs_file = OUTPUT_DIR / "rhi_live_runs_v2.jsonl"
saved_runs = read_saved_runs(runs_file)

summary_rows = []
for r in saved_runs:
    res = r.get("resolution", {})
    winner = res.get("winner") or {}
    summary_rows.append({
        "run_id": r.get("run_id"),
        "time": r.get("time"),
        "state": res.get("state"),
        "reason": res.get("reason"),
        "support": res.get("support"),
        "margin": res.get("margin"),
        "psi": winner.get("psi"),
        "audit_score": winner.get("audit_score"),
        "branch": winner.get("branch"),
        "elapsed_sec": r.get("elapsed_sec"),
        "prompt": str(r.get("prompt", ""))[:180],
    })

runs_df = pd.DataFrame(summary_rows)
display(runs_df)

manifest = {
    "notebook": "rhi_live_runtime_v2_operational_critic",
    "model_name": MODEL_NAME,
    "slot_adapter_dir": str(SLOT_ADAPTER_DIR),
    "output_dir": str(OUTPUT_DIR),
    "runs_file": str(runs_file),
    "n_saved_runs": len(saved_runs),
    "runtime_shape": "Q -> C_sanitized -> {A_i} -> five-dimensional audit -> Ψ/Ω",
    "critic_mode": CRITIC_MODE,
    "collapse_controls": {
        "support_min": SUPPORT_MIN,
        "margin_min": MARGIN_MIN,
        "psi_min": PSI_MIN,
        "audit_min": AUDIT_MIN,
    },
}

(OUTPUT_DIR / "rhi_live_runtime_v2_manifest.json").write_text(
    json.dumps(manifest, indent=2),
    encoding="utf-8",
)

print(json.dumps(manifest, indent=2))


,run_id,time,state,reason,support,margin,psi,audit_score,branch,elapsed_sec,prompt
0,rhi_v2_8ec3b3b6f2,2026-05-04T03:23:57,Ω,support_below_min | margin_below_min | psi_bel...,1,0.000157,0.158157,0.0,direct,236.330702,"Using the Nexus lens, explain why current AI a..."


{
  "notebook": "rhi_live_runtime_v2_operational_critic",
  "model_name": "Qwen/Qwen2.5-1.5B-Instruct",
  "slot_adapter_dir": "D:\\@User Data\\Downloads\\slot_builder_lora_v2",
  "output_dir": "D:\\@User Data\\Downloads\\rhi_live_runtime_v2_outputs",
  "runs_file": "D:\\@User Data\\Downloads\\rhi_live_runtime_v2_outputs\\rhi_live_runs_v2.jsonl",
  "n_saved_runs": 1,
  "runtime_shape": "Q -> C_sanitized -> {A_i} -> five-dimensional audit -> \u03a8/\u03a9",
  "critic_mode": "hybrid",
  "collapse_controls": {
    "support_min": 4,
    "margin_min": 0.06,
    "psi_min": 0.58,
    "audit_min": 0.56
  }
}


# Ψ-collapse

This notebook upgrades the shell:

```text
v1 critic: lexical overlap
v2 critic: five-dimensional operational audit
```

The diagnostic is no longer “did the answer reuse the right words?”

The diagnostic is:

```text
Did the answer preserve the operation,
reject the trap,
respect the boundary,
and produce one executable readout?
```

Next fold:

```text
compare v1 vs v2 traces
  ↓
collect Ω / bad collapses
  ↓
convert them into critic training rows
```
